In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import time

In [7]:
BATCH_SIZE = 4096  
LEARNING_RATE = 0.001
EPOCHS = 50
RFF_COMPONENTS = 4096 # Higher dim for better separation of mixed data
RBF_SIGMA = 2.0       # Bandwidth of the kernel
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on: {DEVICE}")

Running on: cuda


In [8]:
def load_and_preprocess():
    try:
        df = pd.read_csv('Merged_IDS.csv')
    except FileNotFoundError:
        print("Error: File not found.")
        return None, None, None, None

    # Drop the source tag (we want the model to learn generic patterns, not dataset specific ones)
    if 'dataset' in df.columns:
        df.drop('dataset', axis=1, inplace=True)
        
    print(f"Dataset Shape: {df.shape}")

    # 1. Separate Target
    y = df['label'].values
    X = df.drop('label', axis=1)

    # 2. Identify Column Types
    # 'proto' and 'service' are strings -> Categorical
    # Everything else -> Numerical
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
    
    print(f"Categorical Features: {cat_cols}")
    print(f"Numerical Features: {num_cols}")

    # 3. Preprocessing Pipeline
    # StandardScaler is CRITICAL here to align the distributions of the merged features
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ]
    )

    print("Fitting preprocessor (Aligning distributions)...")
    X_enc = preprocessor.fit_transform(X).astype(np.float32)

    # 4. Train/Test Split
    # We shuffle thoroughly to mix NSL and UNSW samples
    X_train, X_test, y_train, y_test = train_test_split(
        X_enc, y, test_size=0.2, random_state=42, shuffle=True
    )
    
    return X_train, y_train, X_test, y_test

In [9]:
class RBFSVM(nn.Module):
    def __init__(self, input_dim, rff_dim=RFF_COMPONENTS, sigma=RBF_SIGMA):
        super(RBFSVM, self).__init__()
        
        # Fixed RFF weights (Gaussian sampling)
        self.rff_weights = nn.Parameter(torch.randn(input_dim, rff_dim) / sigma, requires_grad=False)
        self.rff_bias = nn.Parameter(torch.rand(rff_dim) * 2 * np.pi, requires_grad=False)
        
        # Learnable Linear Layer
        self.fc = nn.Linear(rff_dim, 1)
        
    def forward(self, x):
        # 1. Map to high-dim space: cos(Wx + b)
        x_mapped = torch.cos(x @ self.rff_weights + self.rff_bias)
        # 2. Linear Decision
        return self.fc(x_mapped)

In [10]:
    X_train, y_train, X_test, y_test = load_and_preprocess()
    
    if X_train is not None:
        # Prepare Tensors (Labels: -1 / 1 for Hinge Loss)
        y_train_svm = torch.tensor(np.where(y_train == 0, -1, 1), dtype=torch.float32).to(DEVICE)
        X_train_tensor = torch.tensor(X_train).to(DEVICE)
        X_test_tensor = torch.tensor(X_test).to(DEVICE)

        # DataLoader
        train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_svm), 
                                  batch_size=BATCH_SIZE, shuffle=True)

        # Initialize Model
        input_dim = X_train.shape[1]
        model = RBFSVM(input_dim).to(DEVICE)
        
        # Optimizer
        optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)

        print(f"\nStarting RBF SVM Training on {len(X_train)} samples...")
        start_time = time.time()
        
        model.train()
        for epoch in range(EPOCHS):
            total_loss = 0
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = model(batch_X).squeeze()
                
                # Hinge Loss
                loss = torch.mean(torch.clamp(1 - batch_y * outputs, min=0))
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                
            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader):.4f}")

        print(f"Training Time: {time.time() - start_time:.2f}s")

Dataset Shape: (406190, 14)
Categorical Features: ['proto', 'service']
Numerical Features: ['dur', 'sbytes', 'dbytes', 'is_sm_ips_ports', 'trans_depth', 'is_ftp_login', 'ct_dst_src_ltm', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']
Fitting preprocessor (Aligning distributions)...

Starting RBF SVM Training on 324952 samples...
Epoch [10/50], Loss: 0.2786
Epoch [20/50], Loss: 0.2645
Epoch [30/50], Loss: 0.2578
Epoch [40/50], Loss: 0.2548
Epoch [50/50], Loss: 0.2524
Training Time: 207.09s


In [12]:
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor).squeeze()
    predictions = torch.where(outputs >= 0, 1, 0).cpu().numpy()

print("\n--- Combined Dataset Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print(classification_report(y_test, predictions, target_names=['Normal', 'Attack']))


--- Combined Dataset Evaluation ---
Accuracy: 0.8920
              precision    recall  f1-score   support

      Normal       0.89      0.84      0.87     33965
      Attack       0.89      0.93      0.91     47273

    accuracy                           0.89     81238
   macro avg       0.89      0.89      0.89     81238
weighted avg       0.89      0.89      0.89     81238

